In [1]:
import os
import warnings
import logging
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["ABSL_LOG_LEVEL"] = "3"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"
tf.get_logger().setLevel("ERROR")

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"

import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

import sys
sys.path.append('/kaggle/input/datasets/keithmarange/vadym-util/')
sys.path.append('/kaggle/input/cmi-competition-code')

import pandas as pd
import data_utils
import os
import utils
from sklearn.pipeline import Pipeline
from scipy.stats import randint
from skopt.space import Categorical, Integer
from sklearn_genetic.space import Categorical as ECat, Integer as EInt
from sklearn.model_selection import GridSearchCV
from sklearn_genetic import GASearchCV
from sklearn.model_selection import RandomizedSearchCV
from skopt import BayesSearchCV
from sklearn.model_selection import GroupKFold
from skopt.space import Categorical, Integer, Real
from sklearn_genetic.space import Categorical as ECat, Integer as EInt, Continuous as EFloat
from scipy.stats import randint, uniform, loguniform
from sklearn.metrics import f1_score, make_scorer
import numpy as np

from sklearn.model_selection import KFold
import importlib
warnings.filterwarnings('ignore', module='deap')
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

import utils
from sklearn.metrics import accuracy_score, classification_report

from scipy.spatial.transform import Rotation as R

2026-05-14 22:43:04.279256: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778798584.589746      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778798584.674586      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778798585.361767      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778798585.361823      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778798585.361826      23 computation_placer.cc:177] computation placer alr

In [2]:
data_folder = data_utils.find_data_root()

raw_train_df  = pd.read_csv(data_folder / 'train.csv')
raw_test_df   = pd.read_csv(data_folder / 'test.csv')
train_demo_df = pd.read_csv(data_folder / 'train_demographics.csv')
test_demo_df  = pd.read_csv(data_folder / 'test_demographics.csv')

temp_calculations_folder_name = 'temp_calculations/'
model_run_folder_name = 'model_runs/'
os.makedirs(temp_calculations_folder_name, exist_ok=True)
os.makedirs(model_run_folder_name, exist_ok=True)

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data


In [3]:
n_splits = 3
cv = GroupKFold(n_splits=n_splits)
model_target = 'orientation'

scoring = None

tof_columns = [f'tof_{i}_v{j}' for i in range(1, 6) for j in range(0, 64)]
acc_cols = ['acc_x', 'acc_y', 'acc_z']

pipe_name = "temporal_extractor"
classifier_name = 'CNN_1D'

search_mode = "bayesian"  # grid, random, evolutionary, bayesian

pipe_name = 'temporal_extractor'
candidates = 5
generations = 2
tournament_size = 2
elitism = True
crossover_probability = 0.8
mutation_probability = 0.2

chosen_orientation = None

train_size = 0.2

In [4]:
train_df = raw_train_df.set_index("row_id").copy(deep=True)

# ============================================================
# 1. Handedness correction
# ============================================================

# add handedness without breaking row_id index
train_df["handedness"] = train_df["subject"].map(
    train_demo_df.set_index("subject")["handedness"]
)

# 0 = left-handed, 1 = right-handed
left_handed_mask = train_df["handedness"].eq(0)

# acc_x mirror for left-handed subjects
train_df.loc[left_handed_mask, "acc_x"] *= -1.0

# quaternion correction for left-handed subjects
rot_cols = ["rot_w", "rot_x", "rot_y", "rot_z"]

q_wxyz = train_df.loc[left_handed_mask, rot_cols].to_numpy(dtype=float)

# clean invalid / zero quaternions before scipy
q_wxyz = np.nan_to_num(q_wxyz, nan=0.0, posinf=0.0, neginf=0.0)

norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
bad_q = norm.squeeze() == 0

q_wxyz[bad_q] = np.array([1.0, 0.0, 0.0, 0.0])

norm = np.linalg.norm(q_wxyz, axis=1, keepdims=True)
q_wxyz = q_wxyz / norm

# scipy expects xyzw, data is wxyz
q_xyzw = q_wxyz[:, [1, 2, 3, 0]]

euler_xyz = R.from_quat(q_xyzw).as_euler("xyz", degrees=False)

# handedness correction: keep X/roll, flip Y/pitch and Z/yaw
euler_xyz[:, [1, 2]] *= -1.0

q_xyzw_fixed = R.from_euler("xyz", euler_xyz, degrees=False).as_quat()

# convert back xyzw -> wxyz
q_wxyz_fixed = q_xyzw_fixed[:, [3, 0, 1, 2]]

train_df.loc[left_handed_mask, rot_cols] = q_wxyz_fixed

# if angular velocity already exists before SequenceExtractor
if {"ang_vel_x", "ang_vel_y", "ang_vel_z"}.issubset(train_df.columns):
    train_df.loc[left_handed_mask, ["ang_vel_y", "ang_vel_z"]] *= -1.0


# ============================================================
# 2. Upside-down subject fix
# ============================================================

upside_down_mask = train_df["subject"].isin([
    "SUBJ_019262",
    "SUBJ_045235",
])

# acc all axes flipped
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0

# quaternion xyz flipped, w unchanged
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

# if angular velocity already exists before SequenceExtractor
if {"ang_vel_x", "ang_vel_y", "ang_vel_z"}.issubset(train_df.columns):
    train_df.loc[upside_down_mask, ["ang_vel_x", "ang_vel_y", "ang_vel_z"]] *= -1.0


# ============================================================
# 3. Clean helper column and check
# ============================================================

print("left-handed subjects corrected:", train_df.loc[left_handed_mask, "subject"].nunique())
print("left-handed rows corrected:", int(left_handed_mask.sum()))
print("bad/zero quaternions replaced:", int(bad_q.sum()))

print("upside-down subjects corrected:", train_df.loc[upside_down_mask, "subject"].nunique())
print("upside-down rows corrected:", int(upside_down_mask.sum()))

train_df = train_df.drop(columns=["handedness"])

left-handed subjects corrected: 10
left-handed rows corrected: 71352
bad/zero quaternions replaced: 1234
upside-down subjects corrected: 2
upside-down rows corrected: 12257


In [5]:
if train_size is None:
    rows = (
        (train_demo_df["adult_child"] == 1)
        & (train_demo_df["sex"] == 1)
        & (train_demo_df["handedness"] == 1)
    )

    ideal_subject_ids = (
        train_demo_df.loc[rows]
        .sort_values("elbow_to_wrist_cm", ascending=False)["subject"]
        .to_list()
    )

    train_sample_df = train_df.loc[
        train_df["subject"].isin(ideal_subject_ids) & (train_df["sequence_type"] == "Target")
    ].copy()  # ← ALREADY HAVE .copy() HERE - GOOD!
    
    test_sample_df = None

elif train_size == 0:
    some_sequences = train_df["sequence_id"].unique()[10:20]

    train_sample_df = train_df.loc[
        train_df["sequence_id"].isin(some_sequences)
    ].copy()  # ← ADD .copy() HERE
    
    test_sample_df = None

else:
    target_df = train_df[train_df["sequence_type"] == "Target"].copy()
    
    train_sample_df, test_sample_df = data_utils.sample_balanced_split(
        target_df,
        train_pct=train_size,
        test_pct=0.2,
    )
    # Make sure both are copies
    train_sample_df = train_sample_df.copy()
    if test_sample_df is not None:
        test_sample_df = test_sample_df.copy()

# NOW it's safe to modify - use .loc to be explicit
train_sample_df.loc[:, "gesture_position"] = train_sample_df["gesture"].str.split(" - ").str[0]
train_sample_df.loc[:, "gesture_action"] = train_sample_df["gesture"].str.split(" - ").str[-1]
train_sample_df.loc[:, "composite_target"] = (
    train_sample_df["orientation"].astype(str) + "_" +
    train_sample_df["gesture_action"] + "_" +
    train_sample_df["phase"]
)
train_sample_df.loc[:, "phase_target"] = train_sample_df["phase"]

if test_sample_df is not None and not test_sample_df.empty:
    test_sample_df.loc[:, "gesture_position"] = test_sample_df["gesture"].str.split(" - ").str[0]
    test_sample_df.loc[:, "gesture_action"] = test_sample_df["gesture"].str.split(" - ").str[-1]
    test_sample_df.loc[:, "composite_target"] = (
        test_sample_df["orientation"].astype(str) + "_" +
        test_sample_df["gesture_action"] + "_" +
        test_sample_df["phase"]
    )
    test_sample_df.loc[:, "phase_target"] = test_sample_df["phase"]

# Apply orientation filter to train only
if chosen_orientation is not None and train_sample_df is not None:
    train_sample_df = train_sample_df.loc[
        train_sample_df["orientation"].isin(chosen_orientation)
    ].copy()

if train_sample_df is not None:
    print(f"Train sequences: {train_sample_df['sequence_id'].nunique()}")
if test_sample_df is not None:
    print(f"Test sequences: {test_sample_df['sequence_id'].nunique()}")

Train: 648 seqs | 12.7%
Test:  648 seqs  | 12.7%
Train sequences: 648
Test sequences: 648


In [6]:
if search_mode == "bayesian":
    param_space = {
        # --- Preprocessing (locked from best run) ---
        f"{pipe_name}__acc_mode": Categorical(['displacement']),
        f"{pipe_name}__linear_acc_mode": Categorical([ "baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),
        f"{pipe_name}__compute_dt": Categorical([True]),
        f"{pipe_name}__use_highpass_fallback": Categorical([True]),
        f"{pipe_name}__fix_quaternion_sign": Categorical([ True]),
        f"{pipe_name}__standardize": Categorical([ "mean_std"]),
        f"{pipe_name}__include_mask": Categorical([ True]),
        f"{pipe_name}__thm_mode": Categorical([ "centered"]),

        # --- Parameters to vary ---
        # Sampling rate: try lower (acts as smoothing) and original
        f"{pipe_name}__sampling_rate": Categorical([10]),

        # Window size: scale with sampling rate
        # At 10Hz: 20-40 frames = 2-4 seconds
        # At 25Hz: 5-15 frames = 0.2-0.6 seconds
        f"{pipe_name}__window_size": Categorical([30]),

        # Clip value: moderate clipping worked well
        f"{pipe_name}__clip_value": Categorical([50.0]),

        # Interpolation
        f"{pipe_name}__interp_mode": Categorical(["linear"]),

        # Smooth alpha: try a small amount of additional smoothing
        f"{pipe_name}__smooth_alpha": Categorical([None]),

        # Rotation: rot6d worked, but try alternatives at low sampling rate
        f"{pipe_name}__rotation_mode": Categorical(["rot6d"]),

        # TOF: pooled_diff worked, try variants
        f"{pipe_name}__tof_mode": Categorical([ "sensor_stats"]),
        f"{pipe_name}__tof_fill_mode": Categorical([ "far_255"]),

        # --- Classifier ---
        f"{classifier_name}__maxlen": Integer(120, 180),

        f"{classifier_name}__conv_filters": Categorical([
        # Moderate scale-up
            "128-256-256",
            # "128-256-512-512",
            # "256-256-512",
            # "128-256-384-512",
            # "96-192-384-512",
        ]),

        f"{classifier_name}__kernel_sizes": Categorical([
            # "3-3-5",
            # "3-5-5",
            "5-5-5",        
            # "3-3-3-5",
            # "3-3-5-5",

        ]),

        f"{classifier_name}__pool_sizes": Categorical([
            # Most compact
            "2-2-2",           # pool after every conv layer
            # "2-2-2-2",         # for 4-layer nets
            # "2-2-2-none",
            # "3-3-3",           # larger stride, very aggressive
            # "3-2-2",
        ]),

        f"{classifier_name}__dense_units": Categorical([
            "none",
            '16',
            # "64",
        ]),

        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__spatial_dropout": Real(0, 0.7),
        f"{classifier_name}__dropout": Real(0.0, 0.6),
        f"{classifier_name}__learning_rate": Real(1e-6, 1e-5, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([32]),
        f"{classifier_name}__epochs": Categorical([200]),
        f"{classifier_name}__patience": Categorical([20]),
    }

elif search_mode == "evolutionary":
    param_space = {
        f"{pipe_name}__acc_mode": ECat(["raw", "smoothed", "velocity", "displacement", "jerk"]),
        f"{pipe_name}__linear_acc_mode": ECat([None, "baseline"]),
        f"{pipe_name}__use_acc_magnitude": ECat([False, True]),
        f"{pipe_name}__use_linear_acc_magnitude": ECat([False, True]),
        f"{pipe_name}__sampling_rate": ECat([20, 25, 50]),
        f"{pipe_name}__compute_dt": ECat([True]),
        f"{pipe_name}__clip_value": ECat([None, 20.0, 50.0, 100.0]),
        f"{pipe_name}__interp_mode": ECat([None, "linear"]),
        f"{pipe_name}__use_highpass_fallback": ECat([True]),
        f"{pipe_name}__window_size": EInt(3, 21),
        f"{pipe_name}__smooth_alpha": ECat([None, 0.2, 0.5, 0.8]),
        f"{pipe_name}__standardize": ECat([None, "mean_std"]),
        f"{pipe_name}__include_mask": ECat([False]),

        f"{pipe_name}__rotation_mode": ECat([None, "quaternion", "euler", "delta_euler", "angular_velocity", "rot6d"]),
        f"{pipe_name}__fix_quaternion_sign": ECat([True, False]),

        f"{pipe_name}__tof_mode": ECat([None, "pooled", "pooled_diff", "sensor_stats", "pooled_stats"]),
        f"{pipe_name}__tof_fill_mode": ECat(["nan_interpolate", "far_255", "far_500"]),
        f"{pipe_name}__thm_mode": ECat([None, "raw", "diff", "centered", "centered_diff"]),

        f"{classifier_name}__maxlen": EInt(16, 160),
        f"{classifier_name}__padding_value": ECat([-999.0]),
        f"{classifier_name}__conv_filters": ECat(["32", "64", "128", "32-64", "64-64", "64-128", "32-64-128", "64-128-128"]),
        f"{classifier_name}__kernel_sizes": ECat(["3", "5", "7", "3-3", "5-5", "3-5-7", "5-7-9"]),
        f"{classifier_name}__pool_sizes": ECat(["none", "2", "none-none", "2-none", "2-2", "none-none-none", "2-none-none", "2-2-none"]),
        f"{classifier_name}__dense_units": ECat(["none", "32", "64", "128", "64-32", "128-64"]),
        f"{classifier_name}__use_batch_norm": ECat([True, False]),
        f"{classifier_name}__spatial_dropout": EFloat(0.0, 0.3),
        f"{classifier_name}__dropout": EFloat(0.0, 0.5),
        f"{classifier_name}__learning_rate": EFloat(1e-4, 2e-3),
        f"{classifier_name}__batch_size": ECat([16, 32, 64]),
        f"{classifier_name}__epochs": ECat([60, 80, 120]),
        f"{classifier_name}__patience": ECat([8, 12, 20]),
    }

elif search_mode == "random":

    param_space = {
        f"{pipe_name}__acc_mode": ["raw", "smoothed", "velocity", "displacement", "jerk"],
        f"{pipe_name}__linear_acc_mode": [None, "baseline"],
        f"{pipe_name}__use_acc_magnitude": [False, True],
        f"{pipe_name}__use_linear_acc_magnitude": [False, True],
        f"{pipe_name}__sampling_rate": [20, 25, 50],
        f"{pipe_name}__compute_dt": [True],
        f"{pipe_name}__clip_value": [None, 20.0, 50.0, 100.0],
        f"{pipe_name}__interp_mode": [None, "linear"],
        f"{pipe_name}__use_highpass_fallback": [True],
        f"{pipe_name}__window_size": randint(3, 22),
        f"{pipe_name}__smooth_alpha": [None, 0.2, 0.5, 0.8],
        f"{pipe_name}__standardize": [None, "mean_std"],
        f"{pipe_name}__include_mask": [False],

        f"{pipe_name}__rotation_mode": [None, "quaternion", "euler", "delta_euler", "angular_velocity", "rot6d"],
        f"{pipe_name}__fix_quaternion_sign": [False, True],

        f"{pipe_name}__tof_mode": [None, "pooled", "pooled_diff", "sensor_stats", "pooled_stats"],
        f"{pipe_name}__tof_fill_mode": ["nan_interpolate", "far_255", "far_500"],
        f"{pipe_name}__thm_mode": [None, "raw", "diff", "centered", "centered_diff"],

        f"{classifier_name}__maxlen": randint(16, 161),
        f"{classifier_name}__padding_value": [-999.0],
        f"{classifier_name}__conv_filters": ["32", "64", "128", "32-64", "64-64", "64-128", "32-64-128", "64-128-128"],
        f"{classifier_name}__kernel_sizes": ["3", "5", "7", "3-3", "5-5", "3-5-7", "5-7-9"],
        f"{classifier_name}__pool_sizes": ["none", "2", "none-none", "2-none", "2-2", "none-none-none", "2-none-none", "2-2-none"],
        f"{classifier_name}__dense_units": ["none", "32", "64", "128", "64-32", "128-64"],
        f"{classifier_name}__use_batch_norm": [True, False],
        f"{classifier_name}__spatial_dropout": uniform(0.0, 0.3),
        f"{classifier_name}__dropout": uniform(0.0, 0.5),
        f"{classifier_name}__learning_rate": loguniform(1e-4, 2e-3),
        f"{classifier_name}__batch_size": [16, 32, 64],
        f"{classifier_name}__epochs": [60, 80, 120],
        f"{classifier_name}__patience": [8, 12, 20],
    }

elif search_mode == "grid":
    
    param_space = {
    # ===== FEATURE EXTRACTOR (temporal_extractor) =====
    f"{pipe_name}__acc_mode": ["raw", "smoothed", "velocity", "displacement", "jerk"],                    # Proven best
    f"{pipe_name}__linear_acc_mode": ["baseline"],             
    f"{pipe_name}__use_acc_magnitude": [True],                 
    f"{pipe_name}__use_linear_acc_magnitude": [True],          
    f"{pipe_name}__sampling_rate": [10],                       # Preserve temporal detail
    f"{pipe_name}__compute_dt": [True],                        
    f"{pipe_name}__clip_value": [50.0],                        
    f"{pipe_name}__interp_mode": ["linear"],                   
    f"{pipe_name}__use_highpass_fallback": [True],             
    f"{pipe_name}__window_size": [31],                         # Larger context (was 21)
    f"{pipe_name}__smooth_alpha": [0.0],                       # No smoothing preserves high-freq
    f"{pipe_name}__standardize": ["mean_std"],                 
    f"{pipe_name}__include_mask": [False],                     

    f"{pipe_name}__rotation_mode": ["rot6d"],                  # Most expressive (6D continuous)
    # Alternative: ["delta_euler"] also works well
    f"{pipe_name}__fix_quaternion_sign": [True],               

    f"{pipe_name}__tof_mode": ["pooled_diff"],                 
    f"{pipe_name}__tof_fill_mode": ["nan_interpolate"],        
    f"{pipe_name}__thm_mode": ["centered_diff"],               

    # ===== LARGER CNN CLASSIFIER =====
    f"{classifier_name}__maxlen": [100],                       # Longer sequences (was 200)
    f"{classifier_name}__padding_value": [-999.0],             

    # DEEPER + WIDER: Pyramid up to 256 filters
    f"{classifier_name}__conv_filters": ["128-256-256"],       # 3 conv layers, deeper
    f"{classifier_name}__kernel_sizes": ["5-5-5"],             # Larger kernels, consistent
    f"{classifier_name}__pool_sizes": ["2-2-2"],               # Progressive pooling
    
    # Keep minimal dense head (still "none" but with more conv features)
    f"{classifier_name}__dense_units": ["none"],               
    
    f"{classifier_name}__use_batch_norm": [True],              # Critical for deep models
    f"{classifier_name}__spatial_dropout": [0.2],              # Increased for more filters
    f"{classifier_name}__dropout": [0.4],                      # Higher regularization
    
    f"{classifier_name}__learning_rate": [1e-4],               # Lower LR for deeper net
    f"{classifier_name}__batch_size": [16],                    # Keep small for long sequences
    f"{classifier_name}__epochs": [200],                       # More epochs for convergence
    f"{classifier_name}__patience": [25],                      # Patient early stopping
}

In [7]:
importlib.reload(utils)         

pipeline = Pipeline([
    (pipe_name, utils.SequenceExtractor()),
    (classifier_name, utils.KerasCNN1DSequenceClassifier(
        target=model_target
    )),
])

if search_mode == "bayesian":
    search_obj = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=candidates,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=42,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

elif search_mode == "random":
    search_obj = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_space,
        n_iter=candidates,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        random_state=42,
        refit=True,
        return_train_score=True,
        error_score = np.nan
    )

elif search_mode == "evolutionary":
    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    
    search_obj = GASearchCV(
        estimator=pipeline,
        cv=cv,
        scoring=scoring,
        param_grid=param_space,
        population_size=candidates,
        generations=generations,
        tournament_size=tournament_size,
        elitism=elitism,
        crossover_probability=crossover_probability,
        mutation_probability=mutation_probability,
        criteria="max",
        n_jobs=1,
        verbose=True,
        keep_top_k=5,
        return_train_score=True,
        error_score=np.nan
    )

elif search_mode == "grid":
    search_obj = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=scoring,
        cv=cv,
        n_jobs=1,
        verbose=3,
        refit=True,
        return_train_score=True,
        error_score=np.nan
    )

else:
    raise ValueError("search_mode must be one of: 'bayesian', 'random', 'evolutionary', 'grid'")

In [8]:
y = train_sample_df[['sequence_id', model_target]]
groups = train_sample_df['sequence_id']

print(f"--- {search_mode} Search ---")
if search_mode == 'evolutionary':
    search_obj.fit(train_sample_df, y)
else:
    search_obj.fit(train_sample_df, y, groups=groups)

--- bayesian Search ---
Fitting 3 folds for each of 1 candidates, totalling 3 fits


2026-05-14 22:45:03.477536: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


[CV 1/3] END CNN_1D__batch_size=32, CNN_1D__conv_filters=128-256-256, CNN_1D__dense_units=16, CNN_1D__dropout=0.18947975609222922, CNN_1D__epochs=200, CNN_1D__kernel_sizes=5-5-5, CNN_1D__learning_rate=2.2435271778548955e-06, CNN_1D__maxlen=164, CNN_1D__patience=20, CNN_1D__pool_sizes=2-2-2, CNN_1D__spatial_dropout=0.3849747592668489, CNN_1D__use_batch_norm=True, temporal_extractor__acc_mode=displacement, temporal_extractor__clip_value=50.0, temporal_extractor__compute_dt=True, temporal_extractor__fix_quaternion_sign=True, temporal_extractor__include_mask=True, temporal_extractor__interp_mode=linear, temporal_extractor__linear_acc_mode=baseline, temporal_extractor__rotation_mode=rot6d, temporal_extractor__sampling_rate=10, temporal_extractor__smooth_alpha=None, temporal_extractor__standardize=mean_std, temporal_extractor__thm_mode=centered, temporal_extractor__tof_fill_mode=far_255, temporal_extractor__tof_mode=sensor_stats, temporal_extractor__use_acc_magnitude=True, temporal_extractor

In [9]:
# --- 1. Model Prediction & Evaluation ---
best_model = search_obj.best_estimator_
X_test = test_sample_df.copy()

# Get unique ground truth labels per sequence
y_true_seq = (test_sample_df[['sequence_id', model_target]]
              .drop_duplicates('sequence_id')
              .reset_index(drop=True))

y_pred_seq = best_model.predict(X_test)

# Calculate Accuracy
test_accuracy = accuracy_score(y_true_seq[model_target], y_pred_seq)

print(f"--- Final Test Results ---")
print(f"Best CV Score: {search_obj.best_score_:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print("\nClassification Report:\n", classification_report(y_true_seq[model_target], y_pred_seq))

# --- 2. Cleanly Append Test Results to CV Results ---
if hasattr(search_obj, 'cv_results_'):
    # Convert search results to DataFrame
    cv_results_df = pd.DataFrame(search_obj.cv_results_)
    cv_results_df['search_mode'] = search_mode
    cv_results_df['target'] = model_target
    
    # Create a "Final Test" row matching the CV columns
    # We use 'params' to label it and put the accuracy in 'mean_test_score'
    test_result_row = pd.DataFrame({
        'params': ['FINAL_HOLD_OUT_TEST'],
        'mean_test_score': [test_accuracy],
        'std_test_score': [0],
        'rank_test_score': [0]
    })
    
    # Concat results - holes in the table (like split scores) fill with NaN
    final_report_df = pd.concat([cv_results_df, test_result_row], ignore_index=True)
    
    # Save the consolidated report
    file_path = f"{model_run_folder_name}{search_mode}_{classifier_name}_results.csv"
    final_report_df.to_csv(file_path, index=False)
    
    print(f"Results consolidated and saved to: {file_path}")

--- Final Test Results ---
Best CV Score: 0.8279
Test Accuracy: 0.8256

Classification Report:
                                  precision    recall  f1-score   support

                    Lie on Back       0.88      0.87      0.87       156
     Lie on Side - Non Dominant       0.78      0.73      0.76       162
Seated Lean Non Dom - FACE DOWN       0.76      0.83      0.80       169
                Seated Straight       0.89      0.87      0.88       161

                       accuracy                           0.83       648
                      macro avg       0.83      0.83      0.83       648
                   weighted avg       0.83      0.83      0.83       648

Results consolidated and saved to: model_runs/bayesian_CNN_1D_results.csv
